# Housing Price Prediction — Final Project (Python)

Cleans NYC-area Zillow housing data (2016–2017), imputes missing values with `miceforest`, and trains three models: Decision Tree, OLS, and a hyperparameter-tuned Random Forest.

**Run order:** `get_coordinates.ipynb` → this notebook (or skip the first if `geocoded.csv` already exists).

In [31]:
! python --version

Python 3.8.20


In [32]:
# ! pip3 install pandas numpy scikit-learn statsmodels miceforest googlemaps matplotlib                                                                                                                        


In [33]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import miceforest as mf
import statsmodels.api as sm

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, RandomizedSearchCV

np.random.seed(1001)

## 1. Load & Inspect Data

In [34]:
q_housing = pd.read_csv('../final_project/housing_data_2016_2017.csv', low_memory=False)

# Drop first 28 irrelevant columns
q_housing = q_housing.iloc[:, 28:]

print(q_housing.shape)
q_housing.head()

(2230, 27)


,approx_year_built,cats_allowed,common_charges,community_district_num,coop_condo,date_of_sale,dining_room_type,dogs_allowed,fuel_type,full_address_or_zip_code,...,num_half_bathrooms,num_total_rooms,parking_charges,pct_tax_deductibl,sale_price,sq_footage,total_taxes,walk_score,listing_price_to_nearest_1000,url
0,1955.0,no,$767,25.0,co-op,2/16/2016,combo,no,gas,"Flushing NY, 11355",...,NaN,5.0,NaN,NaN,"$228,000",NaN,NaN,82,NaN,NaN
1,1955.0,no,NaN,25.0,co-op,2/16/2016,formal,no,oil,"30-11 Parsons Blvd, Flushing NY, 11354 ( Sold...",...,NaN,4.0,NaN,NaN,"$235,500",890.0,NaN,89,NaN,NaN
2,2004.0,no,$167,24.0,condo,2/17/2016,combo,no,NaN,"102-14 Lewis Ave, Corona NY, 11368",...,NaN,3.0,NaN,NaN,"$137,550",550.0,"$5,500",90,NaN,NaN
3,2002.0,no,$275,25.0,condo,2/17/2016,combo,no,gas,"144-48 Roosevelt Ave, Flushing NY, 11354",...,NaN,5.0,NaN,NaN,"$545,000",NaN,"$2,260",94,NaN,NaN
4,1949.0,yes,NaN,26.0,co-op,2/18/2016,combo,yes,gas,"245-27 76th Ave, Bellerose NY, 11426",...,NaN,4.0,NaN,39.0,"$241,700",675.0,NaN,71,NaN,NaN


## 2. Data Cleaning

In [35]:
# Drop 4 irrelevant columns
q_housing = q_housing.drop(columns=[
    'date_of_sale', 'url', 'model_type',
    'listing_price_to_nearest_1000'
])

# Fix column name typo from original data
q_housing = q_housing.rename(columns={'pct_tax_deductibl': 'pct_tax_deductible'})

# Parse monetary columns: strip $ and ,
for col in ['sale_price', 'maintenance_cost', 'total_taxes', 'parking_charges', 'common_charges']:
    q_housing[col] = pd.to_numeric(
        q_housing[col].astype(str).str.replace(r'[$,]', '', regex=True),
        errors='coerce'
    )

# community_district_num is a school district identifier, not a quantity (categorical - nominal, not ordinal)
q_housing['community_district_num'] = q_housing['community_district_num'].astype('category')

# Move sale_price to last column
cols = [c for c in q_housing.columns if c != 'sale_price'] + ['sale_price']
q_housing = q_housing[cols]

print(q_housing.dtypes)

approx_year_built            float64
cats_allowed                  object
common_charges               float64
community_district_num      category
coop_condo                    object
dining_room_type              object
dogs_allowed                  object
fuel_type                     object
full_address_or_zip_code      object
garage_exists                 object
kitchen_type                  object
maintenance_cost             float64
num_bedrooms                 float64
num_floors_in_building       float64
num_full_bathrooms             int64
num_half_bathrooms           float64
num_total_rooms              float64
parking_charges              float64
pct_tax_deductible           float64
sq_footage                   float64
total_taxes                  float64
walk_score                     int64
sale_price                   float64
dtype: object


In [36]:
# Combine num_half_bathrooms + num_full_bathrooms into a single total_num_bathrooms feature.
# NA half bathrooms are treated as 0 (NA likely means no half bath).
q_housing['total_num_bathrooms'] = (
    q_housing['num_full_bathrooms'] + q_housing['num_half_bathrooms'].fillna(0)
)
q_housing = q_housing.drop(columns=['num_full_bathrooms', 'num_half_bathrooms'])

In [37]:
# Combine common_charges + maintenance_cost into a single monthly_cost feature.
# Co-ops charge maintenance fees; condos charge common charges. Buildings use one or the other,
# so NAs in each represent 0 for that building type.
q_housing['common_charges'] = q_housing['common_charges'].fillna(0)
q_housing['maintenance_cost'] = q_housing['maintenance_cost'].fillna(0)
q_housing['monthly_cost'] = q_housing['common_charges'] + q_housing['maintenance_cost']
# Revert 0s to NaN: a combined value of 0 means neither fee was listed (truly missing)
q_housing['monthly_cost'] = q_housing['monthly_cost'].replace(0, np.nan)
q_housing = q_housing.drop(columns=['common_charges', 'maintenance_cost'])

In [38]:
# Normalize inconsistent yes/no values in pet policy columns
for col in ['cats_allowed', 'dogs_allowed']:
    q_housing[col] = q_housing[col].astype(str).str.lower()

q_housing['cats_allowed'] = q_housing['cats_allowed'].str.replace('yes', 'y').str.replace('no', 'n')
q_housing['dogs_allowed'] = (
    q_housing['dogs_allowed']
    .str.replace('yes', 'y')
    .str.replace('no', 'n')
    .str.replace('y89', 'y')  # 'yes89' -> 'y89' -> 'y'
)

# Some buildings allow only cats or only dogs, so we cannot simply drop one. 
# We merge into a single animal_allowed feature.
print('cats == dogs everywhere:', (q_housing['cats_allowed'] == q_housing['dogs_allowed']).all())

q_housing['animal_allowed'] = np.where(
    (q_housing['cats_allowed'] == 'y') | (q_housing['dogs_allowed'] == 'y'), 'y', 'n'
)
q_housing = q_housing.drop(columns=['cats_allowed', 'dogs_allowed'])

cats == dogs everywhere: False


In [39]:
# Normalize capitalization in fuel_type
q_housing['fuel_type'] = q_housing['fuel_type'].str.replace('Other', 'other')
print(q_housing['fuel_type'].unique())

['gas' 'oil' nan 'other' 'electric' 'none']


In [40]:
# Normalize garage_exists to y/n
garage_yes_vals = {'Underground', 'Yes', 'yes', 'UG', '1', 'eys'}
q_housing['garage_exists'] = q_housing['garage_exists'].apply(
    lambda x: 'y' if x in garage_yes_vals else x
)
# NaN almost certainly means no garage
q_housing['garage_exists'] = q_housing['garage_exists'].fillna('n')

# Engineered feature: no garage AND no parking charges
# If a parking charge is listed but no garage exists, it likely means street parking fees.
# This flag captures buildings with no parking option at all.
q_housing['no_garage_no_parking_charges'] = (
    (q_housing['garage_exists'] == 'n') & (q_housing['parking_charges'].isna())
).astype(int)

print(q_housing['garage_exists'].unique())

['n' 'y']


In [41]:
# Normalize inconsistent kitchen_type spellings
kitchen_map = {
    'eat in':               'eatin',
    'Eat In':               'eatin',
    'Eat in':               'eatin',
    'efficiency kitchene':  'efficiency',
    'efficiency kitchen':   'efficiency',
    'efficiemcy':           'efficiency',
    'efficiency ktchen':    'efficiency',
    'Combo':                'combo',
}
q_housing['kitchen_type'] = q_housing['kitchen_type'].replace(kitchen_map)
print(q_housing['kitchen_type'].unique())

['eatin' 'efficiency' 'combo' nan '1955' 'none']


In [42]:
# Drop total_taxes. Histogram shows anomalously low values; data quality is unreliable
q_housing = q_housing.drop(columns=['total_taxes'])

# Convert approx_year_built to age at time of sale (or approximate current age if unsold)
q_housing['years_old'] = np.where(
    q_housing['sale_price'].isna(),
    2019 - q_housing['approx_year_built'],
    2017 - q_housing['approx_year_built']
)
q_housing = q_housing.drop(columns=['approx_year_built'])

In [43]:
# Convert remaining string columns to pandas Categorical
# (required for miceforest to use classification instead of regression when imputing them)
cat_cols = ['garage_exists', 'coop_condo', 'dining_room_type', 'fuel_type', 'kitchen_type', 'animal_allowed']
for col in cat_cols:
    q_housing[col] = q_housing[col].astype('category')

print(q_housing.dtypes)
q_housing.head(10)

community_district_num          category
coop_condo                      category
dining_room_type                category
fuel_type                       category
full_address_or_zip_code          object
garage_exists                   category
kitchen_type                    category
num_bedrooms                     float64
num_floors_in_building           float64
num_total_rooms                  float64
parking_charges                  float64
pct_tax_deductible               float64
sq_footage                       float64
walk_score                         int64
sale_price                       float64
total_num_bathrooms              float64
monthly_cost                     float64
animal_allowed                  category
no_garage_no_parking_charges       int64
years_old                        float64
dtype: object


,community_district_num,coop_condo,dining_room_type,fuel_type,full_address_or_zip_code,garage_exists,kitchen_type,num_bedrooms,num_floors_in_building,num_total_rooms,parking_charges,pct_tax_deductible,sq_footage,walk_score,sale_price,total_num_bathrooms,monthly_cost,animal_allowed,no_garage_no_parking_charges,years_old
0,25.0,co-op,combo,gas,"Flushing NY, 11355",n,eatin,2.0,6.0,5.0,NaN,NaN,NaN,82,228000.0,1.0,767.0,n,1,62.0
1,25.0,co-op,formal,oil,"30-11 Parsons Blvd, Flushing NY, 11354 ( Sold...",n,eatin,1.0,7.0,4.0,NaN,NaN,890.0,89,235500.0,1.0,604.0,n,1,62.0
2,24.0,condo,combo,NaN,"102-14 Lewis Ave, Corona NY, 11368",n,efficiency,1.0,1.0,3.0,NaN,NaN,550.0,90,137550.0,1.0,167.0,n,1,13.0
3,25.0,condo,combo,gas,"144-48 Roosevelt Ave, Flushing NY, 11354",n,eatin,3.0,NaN,5.0,NaN,NaN,NaN,94,545000.0,2.0,275.0,n,1,15.0
4,26.0,co-op,combo,gas,"245-27 76th Ave, Bellerose NY, 11426",n,eatin,2.0,2.0,4.0,NaN,39.0,675.0,71,241700.0,1.0,660.0,y,1,68.0
5,28.0,co-op,combo,oil,"124-16 84 Rd, Kew Gardens NY, 11415",n,eatin,2.0,6.0,4.0,NaN,NaN,1000.0,90,250000.0,1.0,932.0,y,1,79.0
6,29.0,co-op,combo,gas,"197-09 Dunton Ave, Holliswood NY, 11423",n,efficiency,1.0,NaN,3.0,NaN,NaN,NaN,72,145000.0,1.0,660.0,n,1,67.0
7,28.0,co-op,NaN,gas,"76-26 113 Street, Forest Hills, NY 11375",n,efficiency,0.0,2.0,2.0,NaN,NaN,375.0,93,109000.0,1.0,514.0,n,1,57.0
8,25.0,co-op,NaN,oil,"209-15 18th Ave, Bayside NY, 11360",n,eatin,1.0,NaN,4.0,20.0,NaN,NaN,70,212000.0,1.0,781.0,n,0,57.0
9,30.0,condo,other,NaN,"32-86 41st St, Astoria NY, 11102",n,eatin,1.0,4.0,3.0,NaN,NaN,681.0,98,535000.0,1.0,NaN,n,1,12.0


In [44]:
# Keep only homes with a recorded sale price. Unsupervised rows are unusable for training
q_housing = q_housing[q_housing['sale_price'].notna()].reset_index(drop=True)
print(f'Rows remaining: {len(q_housing)}')

# Filtering rows leaves orphaned category levels that were only present in unsold homes.
# miceforest raises a TypeError if any categorical column has unused levels, so drop them.
for col in q_housing.select_dtypes(include='category').columns:
    q_housing[col] = q_housing[col].cat.remove_unused_categories()

Rows remaining: 528


## 3. Add Geospatial Features

Lat/lon from `geocoded.csv` (generated by `get_coordinates.ipynb` using the Google Maps API).

In [45]:
geocoded = pd.read_csv('../final_project/geocoded.csv')

# Join on address string ensures coordinates are matched to the correct listing even if row order ever 
# differs between files.
q_housing = q_housing.merge(geocoded, on='full_address_or_zip_code', how='left')
q_housing = q_housing.drop(columns=['full_address_or_zip_code'])

print(q_housing[['lon', 'lat']].describe())

              lon         lat
count  710.000000  710.000000
mean   -73.820682   40.738382
std      0.052171    0.028768
min    -73.957205   40.658407
25%    -73.849753   40.722180
50%    -73.830712   40.737542
75%    -73.782007   40.758872
max    -73.702441   40.795314


## 4. Missingness Imputation (miceforest)

Strategy:
1. Build a binary missingness indicator matrix `M` for columns that have missing values.
2. Impute `X` with `miceforest` using LightGBM under the hood.
3. Combine imputed `X` with `M` and the engineered feature `sqft_per_room` to form `Xnew`.

`miceforest` uses MICE (Multiple Imputation by Chained Equations) and automatically applies LightGBM classification for `Categorical` columns and LightGBM regression for numeric columns.

In [46]:
np.random.seed(1001)

q_housing1 = q_housing.copy()
y = q_housing1['sale_price']

# Keep sale_price in X so it can inform imputation of other columns
X = q_housing1.copy()

# Build binary missingness indicator matrix
M = X.isna().astype(int)
M.columns = ['is_missing_' + c for c in X.columns]
M = M.loc[:, M.sum() > 0]  # Only keep columns that actually have missing values
print(M.sum().sort_values(ascending=False))

is_missing_pct_tax_deductible        581
is_missing_parking_charges           529
is_missing_sq_footage                439
is_missing_num_floors_in_building    163
is_missing_dining_room_type          152
is_missing_fuel_type                  30
is_missing_monthly_cost               25
is_missing_kitchen_type                6
is_missing_years_old                   6
is_missing_community_district_num      1
dtype: int64


In [47]:
missing_cols_to_keep = [
    'is_missing_monthly_cost',
    'is_missing_dining_room_type',
    'is_missing_fuel_type',
    'is_missing_kitchen_type',
    'is_missing_num_floors_in_building',
]
M = M[missing_cols_to_keep]
print(M.sum())

# Drop sparse/low-quality columns before imputation
X = X.drop(columns=['parking_charges', 'pct_tax_deductible'])

is_missing_monthly_cost               25
is_missing_dining_room_type          152
is_missing_fuel_type                  30
is_missing_kitchen_type                6
is_missing_num_floors_in_building    163
dtype: int64


In [48]:
# Impute synthetic data in missing data
kernel = mf.ImputationKernel(X, random_state=1001)
kernel.mice(3)  # 3 iterations of chained equations
Ximp = kernel.complete_data()
print(f'Missing values after imputation: {Ximp.isnull().sum().sum()}')

Missing values after imputation: 0


In [49]:
Xnew = pd.concat(
    [Ximp.reset_index(drop=True), M.reset_index(drop=True)],
    axis=1
)
Xnew['sqft_per_room'] = Xnew['sq_footage'] / Xnew['num_total_rooms']
Xnew.head(10)

,community_district_num,coop_condo,dining_room_type,fuel_type,garage_exists,kitchen_type,num_bedrooms,num_floors_in_building,num_total_rooms,sq_footage,...,no_garage_no_parking_charges,years_old,lon,lat,is_missing_monthly_cost,is_missing_dining_room_type,is_missing_fuel_type,is_missing_kitchen_type,is_missing_num_floors_in_building,sqft_per_room
0,25.0,co-op,combo,gas,n,eatin,2.0,6.0,5.0,922.0,...,1,62.0,-73.821321,40.749310,0,0,0,0,0,184.400000
1,25.0,co-op,combo,gas,n,eatin,2.0,6.0,5.0,922.0,...,1,62.0,-73.821321,40.749310,0,0,0,0,0,184.400000
2,25.0,co-op,formal,oil,n,eatin,1.0,7.0,4.0,890.0,...,1,62.0,-73.824170,40.770161,0,0,0,0,0,222.500000
3,24.0,condo,combo,gas,n,efficiency,1.0,1.0,3.0,550.0,...,1,13.0,-73.858148,40.740986,0,0,1,0,0,183.333333
4,25.0,condo,combo,gas,n,eatin,3.0,3.0,5.0,980.0,...,1,15.0,-73.820258,40.761920,0,0,0,0,1,196.000000
5,26.0,co-op,combo,gas,n,eatin,2.0,2.0,4.0,675.0,...,1,68.0,-73.725316,40.744185,0,0,0,0,0,168.750000
6,28.0,co-op,combo,oil,n,eatin,2.0,6.0,4.0,1000.0,...,1,79.0,-73.827554,40.707722,0,0,0,0,0,250.000000
7,28.0,co-op,combo,oil,n,eatin,2.0,6.0,4.0,1000.0,...,1,79.0,-73.827554,40.707722,0,0,0,0,0,250.000000
8,29.0,co-op,combo,gas,n,efficiency,1.0,3.0,3.0,610.0,...,1,67.0,-73.765936,40.719851,0,0,0,0,1,203.333333
9,28.0,co-op,combo,gas,n,efficiency,0.0,2.0,2.0,375.0,...,1,57.0,-73.833682,40.717247,0,1,0,0,0,187.500000


## 5. Train/Test Split & Data Preparation

Before training any model we split the data and impute train/test sets **separately** to prevent leakage. The test set never influences training-set imputation. The encoded matrices `X_train_enc`, `y_train`, `X_test_enc`, and `y_test` are shared across all three models, enabling fair apples-to-apples comparisons.

In [50]:
# We start from q_housing (the cleaned but unimputed data) rather than Xnew.
# This is intentional: Xnew was imputed on the full dataset, so using it here
# would mean the test set influenced the training set's imputation (data leakage).
# By going back to the raw cleaned data and imputing train/test separately below,
# the test set is never seen during training.
np.random.seed(1001)

# Drop the same sparse/low-quality columns removed before building Xnew
q_housing2 = q_housing.drop(columns=['parking_charges', 'pct_tax_deductible'])

# 80/20 split (train/test)
train_set, test_set = train_test_split(q_housing2, test_size=1/5, random_state=1001)
train_set = train_set.reset_index(drop=True)
test_set  = test_set.reset_index(drop=True)

# The split can leave unused category levels in each subset (a level that only
# appears in train won't appear in test, and vice versa). miceforest rejects
# unused levels, so we drop them from both sets now.
for col in train_set.select_dtypes(include='category').columns:
    train_set[col] = train_set[col].cat.remove_unused_categories()
for col in test_set.select_dtypes(include='category').columns:
    test_set[col] = test_set[col].cat.remove_unused_categories()

print(f'Train: {len(train_set)} rows,  Test: {len(test_set)} rows')

Train: 568 rows,  Test: 142 rows


In [51]:
# --- Prepare training set ---
# sale_price is intentionally LEFT IN during train imputation.
# miceforest can use it as a predictor when filling in other missing columns
# (e.g., a missing monthly_cost can be better estimated if we know the sale price).
# This is safe here because we're only imputing training data.

# Step 1: Build missingness indicators before imputing (captures which values were originally missing)
M_train = train_set.isna().astype(int)
M_train.columns = ['is_missing_' + c for c in train_set.columns]
M_train = M_train.loc[:, M_train.sum() > 0]          # drop all-zero indicator columns
M_train = M_train.reindex(columns=missing_cols_to_keep, fill_value=0)  # keep the same 5 as Xnew

# Step 2: Impute missing values in the training set
kernel_train = mf.ImputationKernel(train_set, random_state=1001)
kernel_train.mice(3)
Ximp_train = kernel_train.complete_data()

# Step 3: Assemble train_set_new = imputed features + missingness indicators + engineered feature
train_set_new = pd.concat([Ximp_train.reset_index(drop=True), M_train.reset_index(drop=True)], axis=1)
train_set_new['sqft_per_room'] = train_set_new['sq_footage'] / train_set_new['num_total_rooms']

# Step 4: Separate the target variable out of the feature matrix
y_train = train_set_new.pop('sale_price')
print(f'Train set shape: {train_set_new.shape}')

Train set shape: (568, 24)


In [52]:
# --- Prepare test set ---
# sale_price is removed BEFORE test imputation to prevent leakage.
# If sale_price were present, miceforest could use it to impute other columns,
# effectively giving the model a peek at the answer during prediction.

# Step 1: Pull out the true labels before the test set is touched
y_test = test_set.pop('sale_price').values
X_test = test_set.copy()

# Step 2: Build missingness indicators before imputing
M_test = X_test.isna().astype(int)
M_test.columns = ['is_missing_' + c for c in X_test.columns]
M_test = M_test.loc[:, M_test.sum() > 0]
M_test = M_test.reindex(columns=missing_cols_to_keep, fill_value=0)

# Step 3: Impute missing values in the test set (independently of the train set)
kernel_test = mf.ImputationKernel(X_test, random_state=1001)
kernel_test.mice(3)
Ximp_test = kernel_test.complete_data()

# Step 4: Assemble test_set_new = imputed features + missingness indicators + engineered feature
test_set_new = pd.concat([Ximp_test.reset_index(drop=True), M_test.reset_index(drop=True)], axis=1)
test_set_new['sqft_per_room'] = test_set_new['sq_footage'] / test_set_new['num_total_rooms']

print(f'Test set shape: {test_set_new.shape}')

Test set shape: (142, 24)


In [53]:
# --- Encode categoricals (shared by all three models) ---
# sklearn requires all-numeric input, so we one-hot encode categorical columns.
# drop_first=True drops one dummy per variable to avoid multicollinearity.
cat_train_cols = train_set_new.select_dtypes(include='category').columns.tolist()
X_train_enc = pd.get_dummies(train_set_new, columns=cat_train_cols, drop_first=True)

cat_test_cols = test_set_new.select_dtypes(include='category').columns.tolist()
X_test_enc = pd.get_dummies(test_set_new, columns=cat_test_cols, drop_first=True)
# Align test to train: if a category level only appears in train, the test set won't
# have that dummy column after get_dummies — reindex fills those gaps with 0.
X_test_enc = X_test_enc.reindex(columns=X_train_enc.columns, fill_value=0)

print(f'Features: {X_train_enc.shape[1]}')
print(f'Train rows: {X_train_enc.shape[0]},  Test rows: {X_test_enc.shape[0]}')

Features: 42
Train rows: 568,  Test rows: 142


## 6. Model 1: Decision Tree (Regression)

Trained on the 80% training split and evaluated on the held-out 20% test set.

In [54]:
dt = DecisionTreeRegressor(random_state=1001)
dt.fit(X_train_enc, y_train)

# Hold-out evaluation
y_hat_dt    = dt.predict(X_test_enc)
oos_res_dt  = y_test - y_hat_dt
oos_r2_dt   = 1 - np.sum(oos_res_dt**2) / np.sum((y_test - y_test.mean())**2)
oos_rmse_dt = np.sqrt(np.mean(oos_res_dt**2))

print(f'Hold-out R²:   {oos_r2_dt:.4f}')
print(f'Hold-out RMSE: ${oos_rmse_dt:,.2f}')

feat_imp = pd.Series(dt.feature_importances_, index=X_train_enc.columns).sort_values(ascending=False)
print('\nTop 10 features by importance:')
print(feat_imp.head(10).to_string())

Hold-out R²:   0.7317
Hold-out RMSE: $90,492.49

Top 10 features by importance:
total_num_bathrooms       0.520654
sq_footage                0.115722
coop_condo_condo          0.106898
num_floors_in_building    0.061808
lat                       0.061665
years_old                 0.046048
monthly_cost              0.023952
walk_score                0.018062
num_bedrooms              0.009356
lon                       0.009160


## 7. Model 2: OLS Regression

Using `statsmodels` for a full summary with R², RMSE, p-values, and confidence intervals. Trained on the 80% training split and evaluated on the held-out 20% test set.

In [55]:
X_train_ols = sm.add_constant(X_train_enc.astype(float))
X_test_ols  = sm.add_constant(X_test_enc.astype(float))
X_test_ols  = X_test_ols.reindex(columns=X_train_ols.columns, fill_value=0)

ols_model = sm.OLS(y_train, X_train_ols).fit()

# Hold-out evaluation
y_hat_ols    = ols_model.predict(X_test_ols)
oos_res_ols  = y_test - y_hat_ols
oos_r2_ols   = 1 - np.sum(oos_res_ols**2) / np.sum((y_test - y_test.mean())**2)
oos_rmse_ols = np.sqrt(np.mean(oos_res_ols**2))

print(f'Training R²:   {ols_model.rsquared:.4f}')
print(f'Hold-out R²:   {oos_r2_ols:.4f}')
print(f'Hold-out RMSE: ${oos_rmse_ols:,.2f}')
print()
print(ols_model.summary())

Training R²:   0.8672
Hold-out R²:   0.8068
Hold-out RMSE: $76,803.76

                            OLS Regression Results                            
Dep. Variable:             sale_price   R-squared:                       0.867
Model:                            OLS   Adj. R-squared:                  0.857
Method:                 Least Squares   F-statistic:                     81.63
Date:                Thu, 02 Apr 2026   Prob (F-statistic):          8.76e-202
Time:                        21:12:24   Log-Likelihood:                -7117.2
No. Observations:                 568   AIC:                         1.432e+04
Df Residuals:                     525   BIC:                         1.451e+04
Df Model:                          42                                         
Covariance Type:            nonrobust                                         
                                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------

## 8. Model 3: Random Forest

Uses the same `X_train_enc`/`X_test_enc` prepared in Section 5. Steps:
1. Fit a baseline RF (`n_estimators=100`, `max_features=6`) to establish initial OOB and hold-out performance.
2. Tune `n_estimators`, `max_features`, and `min_samples_leaf` with `RandomizedSearchCV` (50 iterations, 3-fold CV).
3. Retrain the final production model on **all** data using best hyperparameters.

In [56]:
# --- Baseline Random Forest ---
# n_estimators=100 trees, max_features=6 features considered at each split.
# oob_score=True uses out-of-bag samples (the ~37% of rows not drawn for each tree)
# to get a free internal validation score without touching the hold-out test set.
rf_init = RandomForestRegressor(n_estimators=100, max_features=6, random_state=1001, oob_score=True)
rf_init.fit(X_train_enc, y_train)

# --- OOB performance ---
# oob_prediction_ gives the predicted value for each training row using only
# the trees that did NOT train on that row, which is a free internal validation estimate.
oob_res  = y_train - rf_init.oob_prediction_
oob_rmse = np.sqrt(np.mean(oob_res**2))

# --- Hold-out performance ---
y_hat    = rf_init.predict(X_test_enc)
oos_res  = y_test - y_hat
oos_r2   = 1 - np.sum(oos_res**2) / np.sum((y_test - y_test.mean())**2)
oos_rmse = np.sqrt(np.mean(oos_res**2))

# OOB metrics come from training data; hold-out metrics come from the unseen test set.
# Comparing them shows whether the model generalises or is overfitting.
print(f'OOB R²:        {rf_init.oob_score_:.4f}')
print(f'OOB RMSE:      ${oob_rmse:,.2f}')
print(f'Hold-out R²:   {oos_r2:.4f}')
print(f'Hold-out RMSE: ${oos_rmse:,.2f}')

OOB R²:        0.8848
OOB RMSE:      $62,334.86
Hold-out R²:   0.8235
Hold-out RMSE: $73,402.93


In [57]:
# --- Hyperparameter tuning ---
# Instead of trying every combination (grid search), we randomly sample 50 sets
# of hyperparameters and pick the best, which is faster and often just as effective.
# Each candidate is evaluated with 3-fold cross-validation on the training set.
#
# The three hyperparameters being tuned:
#   n_estimators   : number of trees in the forest (more = more stable, slower)
#   max_features   : features considered at each split (controls tree diversity)
#   min_samples_leaf : minimum rows required at a leaf (controls overfitting)
param_dist = {
    'n_estimators':     list(range(10, 451, 10)),  # ntree:    10–450
    'max_features':     list(range(2, 11)),         # mtry:     2–10
    'min_samples_leaf': list(range(2, 51)),         # nodesize: 2–50
}

rf_base = RandomForestRegressor(random_state=1001)
rf_search = RandomizedSearchCV(
    rf_base,
    param_distributions=param_dist,
    n_iter=50,          # number of random candidates to try
    cv=3,               # 3-fold cross-validation
    scoring='r2',       # optimise for R²
    random_state=1001,
    n_jobs=-1,          # use all available CPU cores
    verbose=1,
)
rf_search.fit(X_train_enc, y_train)

print('Best params:', rf_search.best_params_)
print(f'Best CV R²:  {rf_search.best_score_:.4f}')

Fitting 3 folds for each of 50 candidates, totalling 150 fits
Best params: {'n_estimators': 390, 'min_samples_leaf': 2, 'max_features': 10}
Best CV R²:  0.8545


In [58]:
# --- Final production model ---
# Now that we know the best hyperparameters, we retrain on ALL available data (Xnew),
# not just the 80% training split. More data generally means a better model.
# There is no leakage concern here because we are no longer holding out a test set.
# This model is intended for making predictions on new, unseen listings.
#
# OOB score gives a rough performance estimate even without a hold-out set,
# since each tree is validated on the ~37% of rows it didn't see during training.
cat_feature_cols = Xnew.select_dtypes(include='category').columns.tolist()
X_all_enc = pd.get_dummies(Xnew.drop(columns=['sale_price']), columns=cat_feature_cols, drop_first=True)
y_all = Xnew['sale_price']

rf_final = RandomForestRegressor(
    **rf_search.best_params_,
    random_state=1001,
    oob_score=True,
)
rf_final.fit(X_all_enc, y_all)

print(f'Final model OOB R²: {rf_final.oob_score_:.4f}')
print('Final production model trained on all data — ready for predictions.')

Final model OOB R²: 0.8963
Final production model trained on all data — ready for predictions.
